In [ ]:
"""
Raw data
↓
Inspect data
↓
Identify X / y
↓
Train/test split
↓
Handle missing values
↓
Encode categorical features
↓
Scale numerical features
↓
Train model (fit)
↓
Predict
↓
Evaluate

"""

In [43]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Step 1：自己创建一个“小脏数据集”
df = pd.DataFrame({
    "age": [22, 35, 47, 51, np.nan, 29, 41, 33, 38, 45],
    "income": [30000, 52000, 61000, np.nan, 48000,
               39000, 58000, 45000, 50000, 65000],
    "city": ["Tokyo", "Osaka", "Tokyo", "Chiba", "Osaka",
             "Tokyo", "Chiba", "Osaka", "Tokyo", "Chiba"],
    "bought": [0, 1, 1, 1, 0, 0, 1, 0, 1, 1]
})

print(df.head())
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())


# Step 2：X/y + Train/Test Split

# X = 输入 features
X = df.drop(columns=["bought"])
# y = prediction target
y = df["bought"]

# 先 split，再 impute / encode / scale (防止 data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=1,
    stratify=y
)



# Step 3：处理Missing Values

# 只保留数值型列
X_train_num = X_train.select_dtypes(include=["number"])
X_test_num = X_test.select_dtypes(include=["number"])

# create imputer, 使用 SimpleImputer 来处理缺失值
imputer = SimpleImputer(strategy="mean")


# 只在训练集上fit, 真正学的是training-set feature means
imputer.fit(X_train_num)  

# transform X_train
X_train_imputed = imputer.transform(X_train_num)

# transform X_test
X_test_imputed = imputer.transform(X_test_num)

# 检查是否还有缺失值
np.isnan(X_train_imputed).sum()  


# Step 4：Categorical Encoding

ohe = OneHotEncoder(handle_unknown="ignore")  # sparse=False to get dense array
ohe.fit(X_train[["city"]])  # 只在训练集上fit
ohe.transform(X_train[["city"]]).toarray()  
ohe.transform(X_test[["city"]]).toarray()  
X_train_encoded = ohe.transform(X_train[["city"]]).toarray()
X_test_encoded = ohe.transform(X_test[["city"]]).toarray()


# Step 5：Scaling

scaler = StandardScaler()
#fit numerical X_train
scaler.fit(X_train_imputed)  
# transform numerical X_train
X_train_scaled = scaler.transform(X_train_imputed)
# transform numerical X_test
X_test_scaled = scaler.transform(X_test_imputed)

# Step 6：组合 Features + Logistic Regression
X_train_processed = np.hstack([X_train_imputed, X_train_encoded])
X_test_processed = np.hstack([X_test_imputed, X_test_encoded])  

model = LogisticRegression()
model.fit(X_train_processed, y_train)
y_pred = model.predict(X_test_processed)   
print("Predictions:", y_pred)

# Step 7：Evaluate
accuracy_score = model.score(X_test_processed , y_test)
print("Accuracy:", accuracy_score)



    age   income   city  bought
0  22.0  30000.0  Tokyo       0
1  35.0  52000.0  Osaka       1
2  47.0  61000.0  Tokyo       1
3  51.0      NaN  Chiba       1
4   NaN  48000.0  Osaka       0
(10, 4)
age       float64
income    float64
city          str
bought      int64
dtype: object
age       1
income    1
city      0
bought    0
dtype: int64
Predictions: [1 1 1]
Accuracy: 0.6666666666666666


/Users/dan/ai-learning/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
